In [1]:
import pickle
import sys
import copy
import time
import os

import cobra
import sympy
import pandas as pd
import numpy as np

from tqdm import tqdm

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params
from macromolecules.macromolecule import Macromolecule
from expression.build_me_model import flatten_list
from utils.parameters import human_model as m_model

No objective coefficients in model. Unclear what should be optimized


In [2]:
mu_val = 1e-9
n_cores = 15
counter = 8

lp_path = '/data2/hratch/human_me/other/test_lp/'
with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
    tme0 = pickle.load(handle)

def add_boundary(m, type = 'sink', tme_new = None):
    '''m is a cobra.metabolite or metabolite ID'''
    
    if tme_new is None:
        with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
            tme_new = pickle.load(handle)
    
    if isinstance(m, cobra.Metabolite): # object
        m_id = m.id
    else: # string
        m_id = m
    
    tme_new.add_boundary(tme_new.metabolites.get_by_id(m_id), type =type)
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    
    fn = '/data2/hratch/human_me/other/test_lp/test_' + type + '.tab'
    if not os.path.isfile(fn):
        with open(fn, 'a+') as f:
            f.write('metabolite_id' + '\t' + 'status' + '\n')
        
        
    with open(fn, 'a+') as f:
        f.write(m_id + '\t' + str(stat.max()) + '\n')
        

def _add_boundary(metabs_, type_ = 'sink', tme_new = None):
    '''Metabs_ is a list of cobra.Metabolite or metabolite IDs'''
    if tme_new is None:
        with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
            tme_new = pickle.load(handle)
    if type(metabs_) != list:
        metabs_ = list(metabs_)

    for m in metabs_:
        if isinstance(m, cobra.Metabolite): # object
            tme_new.add_boundary(tme_new.metabolites.get_by_id(m.id), type =type_)
        else: # string
            tme_new.add_boundary(tme_new.metabolites.get_by_id(m), type =type_)
    
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    return tme_new, sln, stat



In [3]:
# sln0, stat0, _ = tme0.solve_lp(mu_val = mu_val)
# tme1, sln1, stat1 = _add_sink(metabs_ = ['gdp_c'])

# tme_demand, sln_demand, stat_demand = _add_demand(metabs_ = [m.id for m in m_model.metabolites])

In [4]:
# import pandas as pd
# res_df = pd.DataFrame(data = {'reactions': [r.id for r in tme0.reactions]})
# res_df['infeasible'] = res_df.reactions.apply(lambda x: sln0[tme0.reactions.index(x)])
# res_df.set_index(res_df.reactions, drop = True, inplace = True)
# res_df['feasible'] = pd.Series(res_df.index).apply(lambda r_id: sln_gdp_demand[tme_gdp_demand.reactions.index(r_id)]).tolist()
# res_df['feasible - infeasible'] = res_df.feasible - res_df.infeasible
# res_df['abs_diff'] = res_df['feasible - infeasible'].abs()
# res_df.sort_values(by = 'abs_diff', ascending = False, inplace = True)
# res_df.drop(columns = ['feasible - infeasible'], inplace = True)
# res_df['change'] = res_df[['feasible', 'infeasible']].apply(lambda x: 'increasing' if x[0] > x[1] else 'decreasing', axis = 1).tolist()


# # biom.sort_values(by = 'feasible', ascending = False, inplace = True)
# # biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

In [5]:
# tolerance = max([abs(v) for v in tme2.infeasible_reactions(mu_val = mu_val, sln = sln2, stat = stat2, tolerance = 0).values()])
# res = pd.DataFrame(columns = ['flux'])
# for r_id, flux in tme0.infeasible_reactions(mu_val = mu_val, sln = sln0, stat = stat0, tolerance = tolerance).items():
#     res.loc[r_id, 'flux'] = flux
# res['abs_flux'] = res.flux.abs()
# res.sort_values(by = 'abs_flux', ascending = False, inplace = True)
# test_metabs = list()
# for r_id in res.head(10).index:
#     test_metabs += [m.id for m in list(tme0.reactions.get_by_id(r_id).metabolites) if not hasattr(m, 'type')]
# test_metabs = sorted(set([r_id for r_id in test_metabs if 'biomass' not in r_id]))

# from itertools import product
# test = sorted(set(flatten_list([[m.id for m in tme0.reactions.get_by_id(r_id).metabolites if not isinstance(m, Macromolecule)] for r_id in flux_res.head(10).index])))
# test += [i[0] + i[1] for i in list(product(['a', 'c', 'u', 't'], ['mp_c', 'tp_c', 'dp_c']))]



In [6]:
# import multiprocessing
# import gc


# metabs_ = [m.id for m in m_model.metabolites]

# # demands
# print('Start demands')
# pool = multiprocessing.Pool(processes = n_cores)
# try:
#     res = pool.starmap(add_sink, zip(metabs_, ['demand']*len(metabs_)))
#     pool.close()
#     pool.join()
#     gc.collect()
# except:
#     pool.close()
#     pool.join()
#     gc.collect()
#     raise ValueError('Parallelization failed')

# #sinks 
# # demands
# print('Start sinks')
# pool = multiprocessing.Pool(processes = n_cores)
# try:
#     res = pool.starmap(add_sink, zip(metabs_, ['sink']*len(metabs_)))
#     pool.close()
#     pool.join()
#     gc.collect()
# except:
#     pool.close()
#     pool.join()
#     gc.collect()
#     raise ValueError('Parallelization failed')                       

In [7]:
# sp = '/data2/hratch/human_me/other/test_lp/'
# sinks = pd.read_csv(sp + 'test_sink.tab', sep = '\t')
# demands = pd.read_csv(sp + 'test_demand.tab', sep = '\t')

# sinks = sinks.sort_values(by = ['status', 'metabolite_id'], ascending = True).reset_index(drop = True)
# demands = demands.sort_values(by = ['status', 'metabolite_id'], ascending = True).reset_index(drop = True)
# boundary = pd.DataFrame(index = demands.metabolite_id)
# boundary['demands'] = demands.status.tolist()

# sinks.index = sinks.metabolite_id
# sinks = sinks.loc[boundary.index, :]
# boundary['sinks'] = sinks.status.tolist()

1) Don't include coupling of ribosomal degradation when coupling protein degradation

2) Include a gmp_c and/or cmp_c demand reaction when coupling protein degradation

3) In the ribosomal degradation reaction, only degrade the protein components, and leave the RNA components as intact units (ribosome --> amino_acids + rRNAs). Untested, so not sure will be feasible, but I imagine it would.

4) Further figure out why the model is infeasible when directly coupling ribosomal degradation
    #reaction: cmp --> amp (check if blocked); why is 5s_rRNA degradation producing an ATP

In [3]:
sln0, stat0, _ = tme0.solve_lp(mu_val = 1e-9)
tme1, sln1, stat1 = _add_boundary(metabs_ = ['cmp_c', 'gmp_c'], type_ = 'demand')

# scale = 1e-5
# with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
#     tme2 = pickle.load(handle)
# for r in tqdm(tme2.reactions):
#     if hasattr(r, 'type') and 'translation' in r.type:
#         rcp = [m for m in r.metabolites if hasattr(m, 'type') and m.type == 'proxy' and 'enzyme' in m.id][0]
#         metabolites_ = r.metabolites.copy()
#         metabolites_[rcp] = r.metabolites[rcp]/scale
#         r._metabolites = metabolites_
# sln2, stat2, _ = tme2.solve_lp(mu_val = 1e-9)

Getting MINOS parameters...
Done in 97.5161 seconds with status 1
Getting MINOS parameters...
Done in 90.9998 seconds with status 0


In [8]:
res = pd.DataFrame(index = [r.id for r in tme0.reactions])
for col, fm in {'infeasible': [sln0, tme0], 'DM': [sln1, tme1]}.items():#, 'couple': [sln2, tme2]}.items():
    res[col] = pd.Series(res.index).apply(lambda r_id: fm[0][fm[1].reactions.index(r_id)]).tolist()
res['feasible_diff'] = (res.DM - res.infeasible).abs()
res.sort_values(by = 'feasible_diff', ascending = False, inplace = True)


In [18]:
rib_deg_reactions = ['TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc', 
                     'co_TRANSLOC_IMPORTtr_COMPLEX_PROTEASOMAL_DEGRADATIONc', 
                    '5s_rRNA_DEGRADATIONc', 
                    '18s_rrna_degradation_DEGRADATIONc', 
                    '28s_rrna_DEGRADATIONc', 
                    '5_8s_rrna_DEGRADATIONc']
res.loc[rib_deg_reactions, :]

,infeasible,DM,feasible_diff
TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc,5.706778e-19,1.304724e-18,7.340458e-19
co_TRANSLOC_IMPORTtr_COMPLEX_PROTEASOMAL_DEGRADATIONc,6.342595e-19,2.002968e-19,4.339627e-19
5s_rRNA_DEGRADATIONc,-1.113865e-27,-7.760484e-53,1.113865e-27
18s_rrna_degradation_DEGRADATIONc,-1.113865e-27,-5.438790e-53,1.113865e-27
28s_rrna_DEGRADATIONc,0.000000e+00,0.000000e+00,0.000000e+00
5_8s_rrna_DEGRADATIONc,-1.113865e-27,3.984142e-53,1.113865e-27


In [101]:
ir = tme0.infeasible_reactions(mu_val = mu_val, sln = sln0, stat = stat0, tolerance = 0)
res.loc[list(ir.keys()),:].sort_values(by = 'feasible_diff', ascending = False)

,infeasible,DM,couple,feasible_diff
GSNt5le,-6.137386e-40,1.364530e-07,3.134808e-10,1.361395e-07
SRTNt6_2_r_R,-4.036207e-44,3.845177e-09,6.166485e-08,5.781967e-08
C226CPT2,-2.077685e-44,6.842219e-09,5.608862e-08,4.924640e-08
FAOXC81_5Zm_0,-1.430790e-56,6.842219e-09,5.608862e-08,4.924640e-08
FAOXC61_3Zm_0,-1.580657e-56,6.842219e-09,5.608862e-08,4.924640e-08
...,...,...,...,...
HGNC:11184_folded_protein_c_DEUBIQUITINATIONc,-1.053112e-59,-1.053112e-59,-1.053112e-59,0.000000e+00
HGNC:12540_retrotranslocated_unfolded_protein_c_DEUBIQUITINATIONc,-9.227818e-24,0.000000e+00,0.000000e+00,0.000000e+00
HGNC:12562_folded_protein_c_DEUBIQUITINATIONc,-8.586119e-60,0.000000e+00,0.000000e+00,0.000000e+00
HGNC:14430_LYSOSOMAL_DEGRADATIONl,-1.311950e-48,0.000000e+00,0.000000e+00,0.000000e+00


In [27]:
lp_path = '/data2/hratch/human_me/other/test_lp/'
with open(lp_path + 'working_version_' + str(8) + '.pickle', 'rb') as handle:
    tme0 = pickle.load(handle)

lp_path = '/data2/hratch/human_me/other/test_lp/'
with open(lp_path + 'working_version_' + str(9) + '.pickle', 'rb') as handle:
    tme1 = pickle.load(handle)

sln0, stat0, _ = tme0.solve_lp(mu_val = 1e-9)
sln1, stat1, _ = tme1.solve_lp(mu_val = 1e-9)

/home/hratch/Projects/human_me/scripts/core/model.py:306 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Getting MINOS parameters...
Done in 100.17 seconds with status 1
Getting MINOS parameters...
Done in 77.9001 seconds with status 1


In [28]:
tme2, sln2, stat2 = _add_boundary(metabs_ = ['cmp_c', 'gmp_c'], type_ = 'demand', tme_new = tme0)
tme3, sln3, stat3 = _add_boundary(metabs_ = ['cmp_c', 'gmp_c'], type_ = 'demand', tme_new = tme1)





Getting MINOS parameters...
Done in 78.7679 seconds with status 0
Getting MINOS parameters...
Done in 75.5845 seconds with status 0


In [29]:
res = pd.DataFrame(index = [r.id for r in tme0.reactions])
for col, fm in {'nodeg_rrna': [sln0, tme0], 'deg_rrna': [sln1, tme1], 
               'nodeg_rrna_dm': [sln2, tme2], 'deg_rrna_dm': [sln3, tme3]}.items():
    res[col] = pd.Series(res.index).apply(lambda r_id: fm[0][fm[1].reactions.index(r_id)]).tolist()
# res['feasible_diff'] = (res.deg_rrna - res.nodeg_rrna).abs()
# res.sort_values(by = 'feasible_diff', ascending = False, inplace = True)

rib_deg_reactions = ['TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc', 
                     'co_TRANSLOC_IMPORTtr_COMPLEX_PROTEASOMAL_DEGRADATIONc', 
                    '5s_rRNA_DEGRADATIONc', 
                    '18s_rrna_degradation_DEGRADATIONc', 
                    '28s_rrna_DEGRADATIONc', 
                    '5_8s_rrna_DEGRADATIONc']
res.loc[rib_deg_reactions, :]

,nodeg_rrna,deg_rrna,nodeg_rrna_dm,deg_rrna_dm
TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc,5.706777e-19,5.706802e-19,9.472343e-19,1.100400e-18
co_TRANSLOC_IMPORTtr_COMPLEX_PROTEASOMAL_DEGRADATIONc,6.342592e-19,6.342651e-19,4.700910e-19,1.690069e-19
5s_rRNA_DEGRADATIONc,1.267584e-52,0.000000e+00,-7.677735e-21,1.689759e-22
18s_rrna_degradation_DEGRADATIONc,0.000000e+00,5.752024e-26,-7.677735e-21,1.689759e-22
28s_rrna_DEGRADATIONc,0.000000e+00,0.000000e+00,-7.677735e-21,1.689759e-22
5_8s_rrna_DEGRADATIONc,1.642042e-52,-1.863530e-24,-7.677735e-21,1.689759e-22


In [22]:
tme0.reactions.get_by_id('TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc').reaction

'2.566194761811221e-07 5s_rRNA_DEGRADATIONc_18s_rrna_degradation_DEGRADATIONc_its_1_frag2_rRNA_DEGRADATIONc_28s_rrna_DEGRADATIONc_5_8s_rrna_DEGRADATIONc_3_COMPLEX_enzyme_deg_proxy + 1.49501425582778e5*mu  2.56619476181122e7 5s_rRNA_DEGRADATIONc_18s_rrna_degradation_DEGRADATIONc_its_1_frag2_rRNA_DEGRADATIONc_28s_rrna_DEGRADATIONc_5_8s_rrna_DEGRADATIONc_3_complex_c + 1.671588598378511e-07 DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_enzyme_deg_proxy + 9.87244365228048e6*mu  1.67158859837851e7 DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_complex_c + TRANSLATION_ELONGATIONc_polyub_complex_c + 8006 atp_c + 1814.87689370000 biomass_protein + 3.836930773104541e-13 biomass_rRNA + 23929 h2o_c --> 18s_rrna_c + 28s_rrna_c + 5_8s_rrna_c + 5s_rrna_c + TRANSLATION_ELONGATIONc_COMPLEX_enzyme_deg_proxy + 8006 adp_c + 1231 ala_L_c + 1389 arg_L_c + 557 asn_L_c + 655 asp_L_c + cleaved_polyubiquitin_moiety_protein_c + 249 cys_L_c + 535 

In [23]:
tme1.reactions.get_by_id('TRANSLATION_ELONGATIONc_COMPLEX_PROTEASOMAL_DEGRADATIONc').reaction

'2.566194761811221e-07 5s_rRNA_DEGRADATIONc_18s_rrna_degradation_DEGRADATIONc_its_1_frag2_rRNA_DEGRADATIONc_28s_rrna_DEGRADATIONc_5_8s_rrna_DEGRADATIONc_3_COMPLEX_enzyme_deg_proxy + 1.49501425582778e5*mu  2.56619476181122e7 5s_rRNA_DEGRADATIONc_18s_rrna_degradation_DEGRADATIONc_its_1_frag2_rRNA_DEGRADATIONc_28s_rrna_DEGRADATIONc_5_8s_rrna_DEGRADATIONc_3_complex_c + 1.671588598378511e-07 DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_COMPLEX_enzyme_deg_proxy + 9.87244365228048e6*mu  1.67158859837851e7 DEUBIQUITINATIONc_PROTEASOMAL_DEGRADATIONc_UBIQUITIN_MONOMER_DEGRADATIONc_1_complex_c + TRANSLATION_ELONGATIONc_polyub_complex_c + 1814.87689370000 biomass_protein + 2325.1785940580003 biomass_rRNA + 119 h2o_c --> TRANSLATION_ELONGATIONc_COMPLEX_enzyme_deg_proxy + 8006 adp_c + 1231 ala_L_c + 29 amp_c + 1389 arg_L_c + 557 asn_L_c + 655 asp_L_c + atp_c + cleaved_polyubiquitin_moiety_protein_c + 29 cmp_c + 249 cys_L_c + 535 gln_L_c + 843 glu_L_c + 1182 gly_c + 33 